# Create FRQNT Awards (Fonds de recherche du Québec — Nature et technologies)

Creates FRQNT awards from the official funded-projects CSVs published by FRQNT on donneesquebec.ca CKAN.

**Prerequisites:** run `scripts/local/frqnt_to_s3.py` to download + aggregate + upload first.

**Data source:** donneesquebec.ca CKAN dataset `liste-du-financement-accorde-par-le-fonds-de-recherche-du-quebec-nature-et-technologies` — one CSV resource per fiscal year (2019-2020 through 2023-2024+). The scraper fetches all available years and aggregates by `Dossier` (project ID), summing `Montant_total` across fiscal years.

**S3 location:** `s3a://openalex-ingest/awards/frqnt/frqnt_projects.parquet`

**FRQNT funder in OpenAlex:** funder_id 4320334841 · display_name "Fonds de recherche du Québec – Nature et technologies" · CA.

**Input columns from frqnt_to_s3.py (after by-Dossier aggregation):**
- `funder_award_id` = `Dossier` (the FRQNT internal project ID, e.g. 196577).
- `title` = `Programme` (FRQNT publishes programme names as project titles — there is no separate project title in the CSV; this is the same string as `funder_scheme`).
- `description` = `keywords` (the `Mots_cles` field; no prose abstract in source).
- `amount` = sum of `Montant_total` across all fiscal years for this Dossier (CAD; 0/neg already NULLed §6.7).
- `amount_research` / `amount_indirect` retained in raw for reference.
- `pi_given_name` / `pi_family_name` from `Titulaire` (`Lastname, Firstname` split).
- `institution_name`, `institution_country` (CANADA), `institution_province` (Québec).
- `funder_scheme` = `Programme`; `programme_volet` and `category` retained in raw.
- `start_year` = earliest `Debut_Financement` year across the Dossier's fiscal years.
- `last_fiscal_year` = latest fiscal-year file the Dossier appears in (proxy for end of funding).

**Notes on coverage** (source-bound, not bugs):
- `institution_name` ~32% — most student bursaries (Bourses de 1er cycle/maîtrise/doctorat/postdoc) have no assigned institution in the source.
- `start_year` ~48% — student bursaries don't expose a single start year.
- `amount` ~90% — early-year records sometimes lack a final Montant_total in source.

provenance `frqnt_data_quebec`, priority 185.

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.frqnt_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/frqnt/frqnt_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.frqnt_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.frqnt_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.frqnt_raw LIMIT 5;

## Step 1.6: Funder existence fail-fast

Must return exactly 1 row. If 0, STOP — the funder is missing from `openalex.common.funder`.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320334841;

## Step 2: Create FRQNT Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.frqnt_awards
USING delta
AS
WITH
frqnt_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320334841  -- Fonds de recherche du Québec – Nature et technologies
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
        g.title as display_name,
        g.description as description,
        f.funder_id,
        g.funder_award_id as funder_award_id,
        CASE WHEN TRY_CAST(g.amount AS DOUBLE) > 0 THEN TRY_CAST(g.amount AS DOUBLE) ELSE NULL END as amount,
        CASE WHEN TRY_CAST(g.amount AS DOUBLE) > 0 THEN 'CAD' ELSE NULL END as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name, f.ror_id, f.doi
        ) as funder,
        CASE
            WHEN LOWER(g.funder_scheme) RLIKE '(bourse|stage|formation|relève|maîtrise|doctorat|postdoc|1er cycle|coll[eé]gial)' THEN 'fellowship'
            ELSE 'research'
        END as funding_type,
        g.funder_scheme as funder_scheme,
        'frqnt_data_quebec' as provenance,
        CASE WHEN TRY_CAST(g.start_year AS INT) IS NOT NULL
             THEN CAST(CONCAT(g.start_year, '-01-01') AS DATE) ELSE NULL END as start_date,
        CAST(NULL AS DATE) as end_date,
        TRY_CAST(g.start_year AS INT) as start_year,
        CAST(NULL AS INT) as end_year,
        CASE
            WHEN g.pi_family_name IS NOT NULL OR g.institution_name IS NOT NULL THEN
                struct(
                    g.pi_given_name as given_name,
                    g.pi_family_name as family_name,
                    CAST(NULL AS STRING) as orcid,
                    struct(
                        g.institution_name as name,
                        'Canada' as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation,
                    CAST(NULL AS DATE) as role_start
                )
            ELSE NULL
        END as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>, role_start:DATE
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>, role_start:DATE
        >>) as investigators,
        'https://www.donneesquebec.ca/recherche/dataset/liste-du-financement-accorde-par-le-fonds-de-recherche-du-quebec-nature-et-technologies' as landing_page_url,
        CAST(NULL AS STRING) as doi,
        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM openalex.awards.frqnt_raw g
    CROSS JOIN frqnt_funder f
    WHERE g.funder_award_id IS NOT NULL
      AND TRIM(CAST(g.funder_award_id AS STRING)) != ''
)
SELECT * FROM awards_transformed;

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'frqnt_data_quebec' AND priority = 185;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    185 as priority
FROM openalex.awards.frqnt_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_frqnt_awards FROM openalex.awards.frqnt_awards;

In [ ]:
%sql
SELECT funder_award_id, display_name, funding_type, amount, currency, start_year, lead_investigator.family_name, lead_investigator.affiliation.name
FROM openalex.awards.frqnt_awards LIMIT 10;

In [ ]:
%sql
SELECT funding_type, COUNT(*) as cnt FROM openalex.awards.frqnt_awards GROUP BY funding_type ORDER BY cnt DESC;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt FROM openalex.awards.frqnt_awards WHERE funder_scheme IS NOT NULL GROUP BY funder_scheme ORDER BY cnt DESC LIMIT 20;

In [ ]:
%sql
-- §6.3 completeness query. NOTE: pct_with_institution and pct_with_start expected ~32-48% — student bursaries (Bourses de 1er cycle/maîtrise/doctorat/postdoc) have no assigned institution and no single start_year in source; this is source-bound, not an ingest bug. pct_with_amount expected ~90% (early-year records sometimes lack a final Montant_total).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(lead_investigator.family_name) as has_pi,
    COUNT(lead_investigator.affiliation.name) as has_institution,
    COUNT(start_date) as has_start_date,
    ROUND(try_divide(COUNT(amount) * 100.0, COUNT(*)), 1) as pct_with_amount,
    ROUND(try_divide(COUNT(lead_investigator.affiliation.name) * 100.0, COUNT(*)), 1) as pct_with_institution
FROM openalex.awards.frqnt_awards;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt FROM openalex.awards.frqnt_awards WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year DESC LIMIT 20;

In [ ]:
%sql
SELECT lead_investigator.affiliation.name as institution, COUNT(*) as grant_count
FROM openalex.awards.frqnt_awards WHERE lead_investigator.affiliation.name IS NOT NULL
GROUP BY lead_investigator.affiliation.name ORDER BY grant_count DESC LIMIT 20;